In [6]:
# GLiNER Taxonomy Tuning Notebook
# 
# This notebook explores GLiNER pollution detection on SOBR data:
# 1. Load sample posts from SOBR
# 2. Run GLiNER with different taxonomy configurations
# 3. Visualize detected span distribution
# 4. Calculate "Explicit Recall" against regex baselines

import sys
sys.path.append('..')

In [ ]:
# Import dependencies
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
from collections import Counter
import importlib

# Force reload in case `gliner_detector.py` was edited during this session
import neuro_stylometry.pollution_guard.gliner_detector as gliner_detector
importlib.reload(gliner_detector)

from neuro_stylometry.pollution_guard.gliner_detector import GLiNERDetector, SOBRTaxonomy
from neuro_stylometry.data_engine.dataset import SOBRDataset

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load SOBR Sample Data

Load a sample of posts from the SOBR laptop dataset for experimentation.

In [8]:
# Load SOBR dataset
dataset_path = Path("../artifacts/data/sobr_laptop.arrow")
dataset = SOBRDataset(arrow_path=dataset_path, seed=42)

# Take a sample of 100 posts
sample_size = 100
table = dataset.table.slice(0, sample_size)
posts = table["post"].to_pylist()
post_ids = table["post_id"].to_pylist()

print(f"Loaded {len(posts)} sample posts")
print(f"\nExample post:\n{posts[0][:500]}...")

Loaded 100 sample posts

Example post:
came to be. I believe the "pushback" when a film shows a character of color is mostly fabricated by production companies in order to get cheap publicity. Marketing people on social media with alt accounts publishing "memes" the same moment a trailer comes out (for and against) to create astroturfed polemic, so now their shitty flick with a predictable plot that has nothing to say it's suddenly trending and a cultural icon. Strategy is a specific genre in gaming. All games require strategizing in...


## 2. Initialize GLiNER Detector

Initialize GLiNER with SOBR-specific taxonomy.

In [10]:
# Initialize GLiNER detector
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

detector = GLiNERDetector(
    model_name="urchade/gliner_large-v2.1",
    device=device,
    max_length=512,
    confidence_threshold=0.85,
)

# Check taxonomy
taxonomy = detector.taxonomy
print(f"\nTarget labels: {len(taxonomy.target_labels)}")
print(f"Distractor labels: {len(taxonomy.distractor_labels)}")

Using device: cuda


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

AttributeError: 'UniEncoderSpanGLiNER' object has no attribute 'token_rep_layer'

## 3. Run Pollution Detection

Detect pollution spans in sample posts.

In [ ]:
# Detect pollution spans
print("Detecting pollution spans...")
entities_batch = detector.detect_spans_long(posts, batch_size=2)

# Count total detections
total_detections = sum(len(entities) for entities in entities_batch)
posts_with_detections = sum(1 for entities in entities_batch if entities)

print(f"\nTotal detections: {total_detections}")
print(f"Posts with detections: {posts_with_detections}/{len(posts)} ({posts_with_detections/len(posts):.1%})")

# Show examples
print("\nExample detections:")
for i, entities in enumerate(entities_batch[:5]):
    if entities:
        print(f"\nPost {i}:")
        for entity in entities[:3]:
            print(f"  - {entity['label']}: '{entity['text']}' (conf={entity['score']:.2f})")

## 4. Visualize Label Distribution

Plot the distribution of detected entity types.

In [ ]:
# Collect all detected labels
all_labels = []
for entities in entities_batch:
    for entity in entities:
        all_labels.append(entity['label'])

label_counts = Counter(all_labels)

# Plot distribution
fig, ax = plt.subplots(figsize=(14, 6))
labels = list(label_counts.keys())
counts = list(label_counts.values())

ax.barh(labels, counts, color='steelblue')
ax.set_xlabel('Count', fontsize=12)
ax.set_ylabel('Entity Type', fontsize=12)
ax.set_title('Distribution of Detected Entity Types', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nMost common entity types:")
for label, count in label_counts.most_common(5):
    print(f"  {label}: {count}")

## 5. Calculate Explicit Recall

Compare GLiNER detections against simple regex baseline to measure recall on explicit patterns.

In [ ]:
# Define regex patterns for explicit self-identification
regex_patterns = {
    'age': r'\b(I am|I\'m)\s+(\d{1,2})\s+(years old|yr old|y/o)\b',
    'gender': r'\b(I am|I\'m|as)\s+a\s+(man|woman|male|female|boy|girl)\b',
    'nationality': r'\b(I am|I\'m)\s+([A-Z][a-z]+ish|[A-Z][a-z]+an|from [A-Z][a-z]+)\b',
    'mbti': r'\b(I am|I\'m|as)\s+an?\s+([IE][NS][FT][JP])\b',
}

# Count regex matches
regex_matches = {pattern_name: 0 for pattern_name in regex_patterns}
gliner_detected = {pattern_name: 0 for pattern_name in regex_patterns}

for i, post in enumerate(posts):
    post_entities = entities_batch[i]
    
    for pattern_name, pattern in regex_patterns.items():
        matches = re.findall(pattern, post, re.IGNORECASE)
        if matches:
            regex_matches[pattern_name] += len(matches)
            
            # Check if GLiNER detected any spans overlapping with regex matches
            for entity in post_entities:
                # Simple overlap check (could be more sophisticated)
                if entity['text'].lower() in post.lower():
                    gliner_detected[pattern_name] += 1
                    break

# Calculate recall
print("Explicit Recall (GLiNER vs Regex Baseline):\n")
for pattern_name in regex_patterns:
    regex_count = regex_matches[pattern_name]
    gliner_count = gliner_detected[pattern_name]
    
    if regex_count > 0:
        recall = gliner_count / regex_count
        print(f"  {pattern_name:12s}: {recall:.1%} ({gliner_count}/{regex_count})")
    else:
        print(f"  {pattern_name:12s}: N/A (no regex matches)")

# Overall
total_regex = sum(regex_matches.values())
total_gliner = sum(gliner_detected.values())
if total_regex > 0:
    overall_recall = total_gliner / total_regex
    print(f"\n  Overall:       {overall_recall:.1%} ({total_gliner}/{total_regex})")